In [1]:
#Set project root variables
from pathlib import Path
import sqlite3
import pandas as pd

project_root = Path.cwd().parents[0]
db_path = project_root / "database" / "faers.db"

conn = sqlite3.connect(db_path)

In [2]:

tables = pd.read_sql_query(
"""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""",
conn
)

tables

,name
0,demo
1,drug
2,faers_severity_dataset
3,fluorouracil_filtered
4,indi
5,outc
6,reac
7,rpsr
8,ther


In [3]:
conn.execute("DROP TABLE IF EXISTS fluorouracil_filtered;")
conn.execute("""
CREATE TABLE fluorouracil_filtered AS
SELECT *
FROM drug
WHERE drugname IN (
    SELECT drugname
    FROM drug
    WHERE drugname LIKE '%FLUOROURACIL%'
    GROUP BY drugname
    HAVING COUNT(*) > 10
);
""")


In [5]:
FU_df_filtered = pd.read_sql_query("""
SELECT *
FROM fluorouracil_filtered
""", conn)

FU_df_filtered.shape

(13211, 21)

In [ ]:

# From this cell onward, classes downsampled to balance the dataset for modeling. The most common drug, FLUOROURACIL, 
# has 1,000+ reports (see 5_FU_explore.ipynb), while the least sufficiently sampled common drug, FLUOROURACIL\IRINOTECAN\LEUCOVORIN\OXALIPLATIN, has only 55 reports. 
# To ensure a balanced dataset for modeling, we will downsample the FLUOROURACIL class to 55 reports, matching the size of the least common class with sufficient samples.


# Downsample the data to balance the classes for modeling. Downsample FLUOROURACIL to the size of FLUOROURACIL\IRINOTECAN\LEUCOVORIN\OXALIPLATIN (55 reports) to ensure balance.

# count reports per drug
counts = FU_df_filtered['drugname'].value_counts()

# keep only groups with >=55 reports
valid_drugs = counts[counts >= 55].index
filtered_df = FU_df_filtered[FU_df_filtered['drugname'].isin(valid_drugs)]

# downsample each group to 55
balanced_df = (
    filtered_df
    .groupby("drugname", group_keys=False)
    .apply(lambda x: x.sample(55, random_state=42))
    .reset_index(drop=True)
)

balanced_df.shape

(275, 20)